1. Are these layers only on drive or also on GEE/Geoserver? 

2. Do style files remain same as the block level style files for Pan India datasets as well? 

3. All layers to be generated or only few? 

In [557]:
!pip install earthengine-api


Defaulting to user installation because normal site-packages is not writeable


In [9]:
import rasterio
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
from shapely.geometry import mapping, box, Polygon
import requests
import xml.etree.ElementTree as ET
from io import BytesIO

import sys
sys.path.append('..')
import constants

import pystac
from pystac.extensions.table import TableExtension
from urllib.parse import urlparse, parse_qs
from pystac import Asset, MediaType
from pystac.extensions.classification import ClassificationExtension, Classification
from pystac.extensions.raster import RasterExtension,RasterBand
from pystac.extensions.projection import ProjectionExtension
import datetime
import ee
ee.Initialize(project='ee-corestackdev')

/home/vishnu/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [12]:


sheet_id = '1rSg8Zm0RHQ7wgyVr7ZZoDj66CB88Om7gbXfqLhDK8HI'
gid='0'
csv_export_url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&gid={gid}'


df = pd.read_csv(csv_export_url)

In [13]:
df

,SNo,Layer Name,GEE asset link,Raster/Vector File,File type and size,Style file url
0,1,Land Use Land Cover,https://code.earthengine.google.com/?asset=pro...,LULC_2017-2018,NaN,https://github.com/core-stack-org/QGIS-Styles/...
1,2,Tree Characteristics,https://code.earthengine.google.com/?asset=pro...,CCD_2017,NaN,https://github.com/core-stack-org/QGIS-Styles/...


In [97]:
layer_name = "Land Use Land Cover"

url_string = df[df["Layer Name"] == layer_name]["GEE asset link"].iloc[0]



In [98]:
url_string

'https://code.earthengine.google.com/?asset=projects/corestack-datasets/assets/datasets/LULC_v3_river_basin/pan_india_lulc_v3_2017_2018'

In [99]:
parsed_url = urlparse(url_string)
query_params = parse_qs(parsed_url.query)

In [100]:
query_params

{'asset': ['projects/corestack-datasets/assets/datasets/LULC_v3_river_basin/pan_india_lulc_v3_2017_2018']}

In [101]:
asset_id = query_params['asset'][0]
print(asset_id)

projects/corestack-datasets/assets/datasets/LULC_v3_river_basin/pan_india_lulc_v3_2017_2018


In [102]:


##assets
#asset_id='projects/corestack-datasets/assets/datasets/Stream_Order_Raster_India'
#asset_id='projects/corestack-datasets/assets/datasets/WRI/LandscapeRestorationOpportunities'
#asset_id='projects/corestack-datasets/assets/datasets/LULC_v3_river_basin/pan_india_lulc_v3_2017_2018'

image = ee.Image(asset_id)
image_info = image.getInfo()

In [103]:
image_info

{'type': 'Image',
 'bands': [{'id': 'predicted_label',
   'data_type': {'type': 'PixelType', 'precision': 'double'},
   'dimensions': [326117, 322830],
   'crs': 'EPSG:4326',
   'crs_transform': [8.983152841195215e-05,
    0,
    68.11403776613543,
    0,
    -8.983152841195215e-05,
    37.0783226781469]}],
 'version': 1752832389901293,
 'id': 'projects/corestack-datasets/assets/datasets/LULC_v3_river_basin/pan_india_lulc_v3_2017_2018',
 'properties': {'system:footprint': {'type': 'LinearRing',
   'coordinates': [[79.78650796613842, 37.078367773348994],
    [76.35344165455938, 37.07836776114237],
    [71.4327133847431, 37.07836773389202],
    [68.11396986781955, 37.07836766249718],
    [68.11399179206582, 8.076693122847033],
    [70.63166458964866, 8.075294360667307],
    [73.95029531922614, 8.073911283840776],
    [77.15449046492586, 8.075691767707651],
    [80.8164278315096, 8.07162220728194],
    [84.70723627834819, 8.074865085320496],
    [87.91143142740033, 8.074865080887003],
   

In [66]:
crs = image_info['bands'][0]['crs']
transform = image_info['bands'][0]['crs_transform']
width = image_info['bands'][0]['dimensions'][0]
height = image_info['bands'][0]['dimensions'][1]
bands = len(image_info['bands'])

In [67]:
min_x = transform[2]
max_y = transform[5]
max_x = min_x + width * transform[0]
min_y = max_y + height * transform[4]
bbox = [min_x, min_y, max_x, max_y]


In [68]:
bbox

[68.11403776613543, 8.078010360916384, 97.40962631725603, 37.0783226781469]

In [69]:
print("CRS:", crs)
print("Bounds:", bbox)
print("Width x Height:", width, "x", height)
print("Bands:", bands)

CRS: EPSG:4326
Bounds: [68.11403776613543, 8.078010360916384, 97.40962631725603, 37.0783226781469]
Width x Height: 326117 x 322830
Bands: 1


In [74]:
DATA_URL = constants.DATA_URL

In [75]:
data_dir = '../data/'
RASTER_STYLE_PATH = '../data/LULC0_12class.qml'


In [104]:
def read_raster_data(image_info):

    crs = image_info['bands'][0]['crs']
    transform = image_info['bands'][0]['crs_transform']
    width = image_info['bands'][0]['dimensions'][0]
    height = image_info['bands'][0]['dimensions'][1]
    bands = len(image_info['bands'])

    min_x = transform[2]
    max_y = transform[5]
    max_x = min_x + width * transform[0]
    min_y = max_y + height * transform[4]

    bbox = [min_x, min_y, max_x, max_y]

    footprint = Polygon([
        [min_x, min_y], 
        [min_x, max_y], 
        [max_x, max_y], 
        [max_x, min_y] 
    ])
    
    gsd = 10
    shape = (bands, height, width)
    data_type = image_info['bands'][0]['data_type']['type']

    return (bbox,mapping(footprint),crs,
            gsd,
            shape,
            data_type
            )

    

In [105]:
bbox,footprint,crs,gsd,shape,data_type = read_raster_data(image_info)

In [106]:
crs

'EPSG:4326'

In [107]:
bbox

[68.11403776613543, 8.078010360916384, 97.40962631725603, 37.0783226781469]

In [112]:
shape 

(1, 322830, 326117)

In [84]:
def create_raster_item(raster_filepath,id):

    # raster_data,bbox,footprint,crs,id,gsd,shape,data_type = read_raster_data(raster_filepath)
    raster_data,bbox,footprint,crs,gsd,shape,data_type = read_raster_data(raster_filepath)

    raster_item = pystac.Item(id=id,
                        geometry=footprint,
                        bbox=bbox,
                        datetime=datetime.datetime.now(datetime.timezone.utc),
                        properties={
                            #   title
                            # description
                            # "gsd": gsd, #adding this in raster extension 
                        })
    
    #add certain metadata under projection extension
    proj_ext = ProjectionExtension.ext(raster_item, add_if_missing=True)
    proj_ext.epsg = crs
    proj_ext.shape = [shape[0], shape[1]]

    return (raster_item,raster_data)


In [108]:
def add_raster_data_asset(raster_item,
                          url_string
                          ):
    raster_item.add_asset("data", Asset(
    # href=os.path.join(data_url, os.path.relpath(raster_path, start=data_dir)), #TODO
    href=url_string,
    roles=["data"],
    title="Raster Layer"))

    return raster_item

In [109]:
def add_raster_extension(raster_item): #TODO
        #add certain metadata under raster extension
    raster_ext = RasterExtension.ext(raster_item.assets["data"], add_if_missing=True)
    raster_band = RasterBand.create(
        data_type=data_type, 
        spatial_resolution=gsd,
        # nodata=nodata
    )
    raster_ext.bands = [raster_band] 

In [40]:
style_file_url='https://raw.githubusercontent.com/core-stack-org/QGIS-Styles/main/Land/LULC0_12class.qml'


In [41]:
os.path.basename(style_file_url)


'LULC0_12class.qml'

In [42]:
def parse_raster_style_file(style_file_url,
                            STYLE_FILE_DIR
                            ):
    
    #download style file if not already downloaded, and save it locally
    #if !os.path.isfile():
    

    tree = ET.parse(style_file_url)
    root = tree.getroot()
    classes = []

    for entry in root.findall(".//paletteEntry"):
        class_info = {}
        for attr_key, attr_value in entry.attrib.items():
            if attr_key == "value":
                try:
                    class_info[attr_key] = int(attr_value)
                except ValueError:
                    class_info[attr_key] = attr_value
            else:
                class_info[attr_key] = attr_value
        classes.append(class_info)

    # If no paletteEntry tags are found, check for item tags
    if not classes:
        for entry in root.findall(".//item"):
            class_info = {}
            for attr_key, attr_value in entry.attrib.items():
                if attr_key == "value":
                    try:
                        class_info[attr_key] = int(attr_value)
                    except ValueError:
                        class_info[attr_key] = attr_value
                else:
                    class_info[attr_key] = attr_value
            classes.append(class_info)
    return classes

In [43]:
def add_classification_extension(raster_style_url,
                                 raster_item
                                 ):
    
    style_info = parse_raster_style_file(style_file_url=raster_style_url)
    classification_ext = ClassificationExtension.ext(raster_item.assets["data"], add_if_missing=True)
    stac_classes = []
    for cls in style_info:
        stac_class_obj = Classification.create(
            value=int(cls["value"]),
            name=cls.get("label") or f"Class {cls['value']}",
            description=cls.get("label"),
            color_hint=cls['color'].replace('#','')
        )
        stac_classes.append(stac_class_obj)
    classification_ext.classes = stac_classes

    return (raster_item,style_info) #style info is required for thumbnail 

In [44]:
def add_stylefile_asset(raster_item,
                        style_file_url):
    raster_item.add_asset("style", Asset(
        # href=os.path.join(data_url, os.path.relpath(raster_style_path, start=data_dir)),
        href=style_file_url,
        media_type=MediaType.XML,
        roles=["metadata"],
        title="QGIS Style file"
    ))
    return raster_item

In [ ]:
def generate_raster_stac(state,
                         district,
                         block,
                         layer_name,
                         layer_map_csv_path,
                         start_year='',
                         end_year=''
                         ):    
    
    # #1. get geoserver url parameters from the layer details
    # geoserver_workspace_name,geoserver_layer_name,style_file_url = \
    #     read_layer_mapping(layer_map_csv_path = layer_map_csv_path,
    #                        district = district,
    #                        block=block,
    #                        layer_name=layer_name,
    #                        start_year=start_year,
    #                        end_year=end_year
    #                        )

    # #2. generate geoserver url
    # geoserver_url = generate_raster_url(workspace=geoserver_workspace_name,
    #                                     layer_name=geoserver_layer_name,
    #                                     geoserver_base_url=GEOSERVER_BASE_URL)
    
    #3. create raster item
    raster_item,raster_data = create_raster_item(geoserver_url,
                                                 id=geoserver_layer_name)
    
    #4. add raster data asset
    raster_item = add_raster_data_asset(raster_item,
                                    asset_id=asset_id)
    
    #5. add classification extension
    raster_item,style_info = add_classification_extension(raster_style_path=style_file_url,
                                                          raster_item=raster_item)
    
    
    

    add_raster_data_asset()
    add_raster_extension()
    parse_raster_style_file()
    add_classification_extension()
    #generate_raster_thumbnail()
    add_stylefile_asset()
    #add_thumbnail_asset()